# Week 2

In [ ]:
# 1. Imports
import pandas as pd
import joblib
import json
from datetime import datetime
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
from datetime import datetime
import numpy as np
from sklearn.preprocessing import StandardScaler
os.makedirs('assets', exist_ok=True)
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [5]:
# -----------------------------
# ✅ Phase 2 Setup: Reload Dataset & Key Variables
# -----------------------------

# Load raw dataset (same as in EDA1)
data_path = "assets/Tetuan City power consumption.csv"
df = pd.read_csv(data_path, parse_dates=["DateTime"])

# Sort by DateTime to ensure temporal order
df = df.sort_values("DateTime").reset_index(drop=True)

# Split out datetime and numeric subsets
datetime_col = df["DateTime"]
numeric_df = df.drop(columns=["DateTime"])

print("✅ Dataset reloaded for Phase 2")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

✅ Dataset reloaded for Phase 2
Shape: (52416, 9)
Columns: ['DateTime', 'Temperature', 'Humidity', 'Wind Speed', 'general diffuse flows', 'diffuse flows', 'Zone 1 Power Consumption', 'Zone 2  Power Consumption', 'Zone 3  Power Consumption']


📌Week 2 | Sequence Generation

In [6]:
# -----------------------------
# ✅ Feature Engineering Prep (before sequence generation)
# -----------------------------
# Copy dataset to new feature dataframe
df_feat = df.copy()

# Extract cyclical time features
df_feat["hour"] = df_feat["DateTime"].dt.hour
df_feat["weekday"] = df_feat["DateTime"].dt.weekday

df_feat["hour_sin"] = np.sin(2 * np.pi * df_feat["hour"] / 24)
df_feat["hour_cos"] = np.cos(2 * np.pi * df_feat["hour"] / 24)
df_feat["weekday_sin"] = np.sin(2 * np.pi * df_feat["weekday"] / 7)
df_feat["weekday_cos"] = np.cos(2 * np.pi * df_feat["weekday"] / 7)

# Keep only numeric features (inputs + targets)
numeric_df = df_feat.select_dtypes(include=[np.number])

# Keep datetime column separately
datetime_col = df_feat["DateTime"].copy()

print("Numeric DF shape:", numeric_df.shape)
print("Datetime col length:", len(datetime_col))

Numeric DF shape: (52416, 14)
Datetime col length: 52416


### Feature Engineering Prep

- Copied the raw DataFrame into a working feature frame.  
- Engineered **cyclical time features** (`hour_sin`, `hour_cos`, `weekday_sin`, `weekday_cos`) using sine/cosine transforms to capture periodicity without sharp boundary jumps (e.g., 23→0 or Sun→Mon).  
- Retained only **numeric features** for modeling while preserving the **DateTime column separately** to ensure proper sequence alignment and artifact tracing downstream.  

In [7]:
# 🔧 Feature Scaling (Standardization)

scaler = StandardScaler()
scaled_array = scaler.fit_transform(numeric_df)

# Convert back to DataFrame with original columns
scaled_df = pd.DataFrame(scaled_array, columns=numeric_df.columns, index=numeric_df.index)

# Save scaler for inference use
scaler_path = os.path.join("assets", "scalers")
os.makedirs(scaler_path, exist_ok=True)
joblib.dump(scaler, os.path.join(scaler_path, "standard_scaler.pkl"))

print("✅ Features standardized and scaler saved at assets/scalers/standard_scaler.pkl")

✅ Features standardized and scaler saved at assets/scalers/standard_scaler.pkl


In [8]:
# -----------------------------
# ✅ Load Phase 1 summary (lookbacks & horizons)
# -----------------------------
assets_dir = "assets"
summary_path = os.path.join(assets_dir, "phase1_eda_summary.json")

with open(summary_path, "r") as f:
    eda_summary = json.load(f)

lookback_options = eda_summary.get("lookbacks", [144])   # fallback: 24h (10-min data)
horizon_options  = eda_summary.get("horizons", [6])      # fallback: 1h ahead

print("Available lookbacks:", lookback_options)
print("Available horizons:", horizon_options)


# -----------------------------
# ✅ Multi-zone Sequence Maker
# -----------------------------
def make_sequences_multizone(data: pd.DataFrame, target_cols: list, lookback: int, horizon: int):
    """
    Generate sequences for multiple target columns (zones).
    
    Args:
        data (pd.DataFrame): Input dataframe with numeric features + targets.
        target_cols (list): List of target column names.
        lookback (int): Number of past timesteps to use as features.
        horizon (int): Number of timesteps ahead to forecast.
    
    Returns:
        X (np.ndarray): Input features (n_samples, lookback, n_features).
        y (np.ndarray): Multi-zone targets (n_samples, horizon, n_targets).
    """
    values = data.values
    target_indices = [data.columns.get_loc(c) for c in target_cols]

    X, y = [], []
    for i in range(len(values) - lookback - horizon + 1):
        X.append(values[i : i + lookback])
        y.append(values[i + lookback : i + lookback + horizon, target_indices])

    return np.array(X), np.array(y)


# -----------------------------
# ✅ DateTime Sequence Maker
# -----------------------------
def make_datetime_sequences(datetime_series, lookback, horizon):
    """
    Generate sequences of DateTime values corresponding to input sequences.
    """
    dt_sequences = []
    n_samples = len(datetime_series) - lookback - horizon + 1
    for i in range(n_samples):
        dt_seq = datetime_series[i:i+lookback].values
        dt_sequences.append(dt_seq)
    return np.array(dt_sequences)


# -----------------------------
# ✅ Generate & Save Sequences (All Zones Together)
# -----------------------------
target_columns = [
    "Zone 1 Power Consumption",
    "Zone 2  Power Consumption",
    "Zone 3  Power Consumption"
]

save_dir = os.path.join(assets_dir, "prepared_sequences")
os.makedirs(save_dir, exist_ok=True)

for lookback in lookback_options:
    for horizon in horizon_options:
        # Generate numeric sequences
        X, y = make_sequences_multizone(
            numeric_df,
            target_cols=target_columns,
            lookback=lookback,
            horizon=horizon
        )

        # Generate corresponding DateTime sequences
        dt_seq = make_datetime_sequences(datetime_col, lookback, horizon)

        # Save sequences
        np.save(os.path.join(save_dir, f"X_allzones_lb{lookback}_hr{horizon}.npy"), X)
        np.save(os.path.join(save_dir, f"y_allzones_lb{lookback}_hr{horizon}.npy"), y)
        np.save(os.path.join(save_dir, f"datetime_lb{lookback}_hr{horizon}.npy"), dt_seq)

        print(
            f"✅ Saved sequences for lb{lookback}, hr{horizon} | "
            f"Shapes -> X: {X.shape}, y: {y.shape}, DateTime: {dt_seq.shape}"
        )

Available lookbacks: [144]
Available horizons: [6]
✅ Saved sequences for lb144, hr6 | Shapes -> X: (52267, 144, 14), y: (52267, 6, 3), DateTime: (52267, 144)


### Sequence Generation

- Loaded Phase 1 lookback/horizon settings to guide sequence construction.  
- Defined multi-zone and DateTime sequence makers to create aligned input/target windows.  
- Generated and saved sequences for all zones under `assets/prepared_sequences`, logging their shapes for verification.  

In [23]:
# -----------------------------
# ✅ Sequence Alignment Validator (Final - CRITICAL FIX 5)
# -----------------------------
def validate_sequence_alignment(X, y, datetime_series, lookback, horizon):
    """
    Validate that sequences are properly aligned and no future information leakage
    """
    print(f"🔍 Validating sequence alignment for lb{lookback}_hr{horizon}.")
    
    validation_results = {
        'total_sequences': len(X),
        'lookback': lookback,
        'horizon': horizon,
        'alignment_errors': 0,
        'future_leakage_errors': 0,
        'timestamp_gaps': []
    }
    
    for i in range(len(X)):
        try:
            input_timestamps = datetime_series[i:i+lookback]
            target_timestamps = datetime_series[i+lookback:i+lookback+horizon]
            
            last_input_time = input_timestamps[-1]
            first_target_time = target_timestamps[0]
            
            if last_input_time >= first_target_time:
                validation_results['future_leakage_errors'] += 1
                print(f"❌ Future leakage at sequence {i}: {last_input_time} >= {first_target_time}")
            
            time_diff = first_target_time - last_input_time
            expected_gap = pd.Timedelta(minutes=10)
            
            if time_diff != expected_gap:
                validation_results['timestamp_gaps'].append({
                    'sequence': i,
                    'actual_gap': str(time_diff),
                    'expected_gap': str(expected_gap)
                })
                print(f"⚠️  Unexpected time gap at sequence {i}: {time_diff} (expected {expected_gap})")
            
            if np.isnan(X[i]).any() or np.isnan(y[i]).any():
                validation_results['alignment_errors'] += 1
                print(f"❌ NaN values detected in sequence {i}")
                
        except Exception as e:
            validation_results['alignment_errors'] += 1
            print(f"❌ Error validating sequence {i}: {e}")
    
    validation_file = os.path.join(save_dir, f"sequence_validation_lb{lookback}_hr{horizon}.json")
    with open(validation_file, 'w') as f:
        json.dump(validation_results, f, indent=2)
    
    if validation_results['alignment_errors'] == 0 and validation_results['future_leakage_errors'] == 0:
        print(f"✅ Sequence alignment validation PASSED for lb{lookback}_hr{horizon}")
        print(f"   Total sequences: {validation_results['total_sequences']}")
        print(f"   Validation results saved: {validation_file}")
    else:
        print(f"❌ Sequence alignment validation FAILED for lb{lookback}_hr{horizon}")
        print(f"   Alignment errors: {validation_results['alignment_errors']}")
        print(f"   Future leakage errors: {validation_results['future_leakage_errors']}")
    
    return validation_results

### 🛠️ CRITICAL FIX 5 – Sequence Alignment Validator
- Added a robust validator to ensure chronological correctness of sequences.  
- Detects future leakage, missing values, and unexpected time gaps (10-minute cadence).  
- Logs detailed validation results to JSON for reproducibility and debugging.

In [24]:
# 🚨 CRITICAL FIX 8: Comprehensive sample count logging
def log_dataset_configuration(lookback, horizon, feature_set, splits, save_dir):
    """
    Log all dataset configuration details for reproducibility
    """
    config_log = {
        'timestamp': str(pd.Timestamp.now()),
        'lookback_window': lookback,
        'prediction_horizon': horizon,
        'feature_set': feature_set,
        'splits': {
            'train': {
                'samples': len(splits['train']),
                'shape': splits['train'].shape,
                'memory_mb': splits['train'].nbytes / (1024 * 1024)
            },
            'val': {
                'samples': len(splits['val']),
                'shape': splits['val'].shape,
                'memory_mb': splits['val'].nbytes / (1024 * 1024)
            },
            'test': {
                'samples': len(splits['test']),
                'shape': splits['test'].shape,
                'memory_mb': splits['test'].nbytes / (1024 * 1024)
            }
        },
        'total_sequences': sum(len(s) for s in splits.values()),
        'total_memory_mb': sum(s.nbytes for s in splits.values()) / (1024 * 1024)
    }

    config_file = os.path.join(save_dir, f"config_log_lb{lookback}_hr{horizon}.json")
    with open(config_file, 'w') as f:
        json.dump(config_log, f, indent=2)

    print(f"📊 Configuration logged: {config_log['total_sequences']} total sequences")
    print(f"   Memory usage: {config_log['total_memory_mb']:.2f} MB")
    print(f"   Config saved: {config_file}")

    return config_log

### 🛠️ CRITICAL FIX 8 – Dataset Configuration Logging
- Introduced comprehensive logging of dataset splits (train/val/test).  
- Captures sample counts, tensor shapes, and memory usage per split.  
- Stores results in JSON config files to guarantee reproducibility across runs.  

In [ ]:
# %% -----------------------------
# ✅ Step 3: Train/Val/Test Split + Tensor Conversion + Safe Scaling
# -----------------------------

# Ratios
train_ratio, val_ratio, test_ratio = 0.7, 0.15, 0.15

seq_dir   = os.path.join(assets_dir, "prepared_sequences")
split_dir = os.path.join(assets_dir, "splits")
scale_dir = os.path.join(assets_dir, "scalers")  # separate folder for clarity
os.makedirs(split_dir, exist_ok=True)
os.makedirs(scale_dir, exist_ok=True)

def normalize_and_save_with_artifacts_fit_on_train(X_train, lookback, horizon, save_dir):
    n_train, lb, n_features = X_train.shape
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train.reshape(-1, n_features)).reshape(n_train, lb, n_features)

    # Save scaler
    scaler_filename = f"scaler_X_allzones_lb{lookback}_hr{horizon}.joblib"
    scaler_filepath = os.path.join(save_dir, scaler_filename)
    joblib.dump(scaler, scaler_filepath)

    # Save scaler metadata
    scaler_metadata = {
        "scaler_type": "StandardScaler",
        "fitted_features": n_features,
        "mean_values": scaler.mean_.tolist(),
        "scale_values": scaler.scale_.tolist(),
        "lookback": lookback,
        "horizon": horizon,
        "timestamp": str(pd.Timestamp.now()),
    }
    metadata_filename = f"scaler_metadata_lb{lookback}_hr{horizon}.json"
    with open(os.path.join(save_dir, metadata_filename), "w") as f:
        json.dump(scaler_metadata, f, indent=2)

    print(f"💾 Scaler saved: {scaler_filepath}")
    print(f"💾 Scaler metadata saved: {os.path.join(save_dir, metadata_filename)}")
    return X_train_scaled, scaler

def transform_with_scaler(X, scaler):
    n, lb, nf = X.shape
    return scaler.transform(X.reshape(-1, nf)).reshape(n, lb, nf)

# Dictionary to hold splits
splits = {}

for lb in lookback_options:
    for hr in horizon_options:
        # Load sequences
        X = np.load(os.path.join(seq_dir, f"X_allzones_lb{lb}_hr{hr}.npy"))
        y = np.load(os.path.join(seq_dir, f"y_allzones_lb{lb}_hr{hr}.npy"))

        n_samples = len(X)
        train_end = int(n_samples * train_ratio)
        val_end   = train_end + int(n_samples * val_ratio)

        # ✅ Save split indices for reproducibility
        split_indices = {
            "lookback": lb,
            "horizon": hr,
            "train_end": train_end,
            "val_end": val_end,
            "train_samples": train_end,
            "val_samples": val_end - train_end,
            "test_samples": n_samples - val_end,
            "total_samples": n_samples,
            "split_ratios": {"train": train_ratio, "val": val_ratio, "test": test_ratio},
        }
        with open(os.path.join(split_dir, f"split_indices_lb{lb}_hr{hr}.json"), "w") as f:
            json.dump(split_indices, f, indent=2)
        print(f"💾 Split indices saved: {os.path.join(split_dir, f'split_indices_lb{lb}_hr{hr}.json')}")

        # Chronological split
        X_train_raw, X_val_raw, X_test_raw = X[:train_end], X[train_end:val_end], X[val_end:]
        y_train,     y_val,     y_test     = y[:train_end], y[train_end:val_end], y[val_end:]

        # ✅ Fit scaler on TRAIN only, then transform all splits; save artifacts
        X_train_scaled, scaler = normalize_and_save_with_artifacts_fit_on_train(
            X_train_raw, lookback=lb, horizon=hr, save_dir=scale_dir
        )
        X_val_scaled  = transform_with_scaler(X_val_raw,  scaler)
        X_test_scaled = transform_with_scaler(X_test_raw, scaler)

        # Convert to tensors
        X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
        y_train_t = torch.tensor(y_train,       dtype=torch.float32)
        X_val_t   = torch.tensor(X_val_scaled,  dtype=torch.float32)
        y_val_t   = torch.tensor(y_val,         dtype=torch.float32)
        X_test_t  = torch.tensor(X_test_scaled, dtype=torch.float32)
        y_test_t  = torch.tensor(y_test,        dtype=torch.float32)

        # Save NumPy splits (scaled)
        key = f"lb{lb}_hr{hr}"
        np.save(os.path.join(split_dir, f"X_train_{key}.npy"), X_train_scaled)
        np.save(os.path.join(split_dir, f"y_train_{key}.npy"), y_train)
        np.save(os.path.join(split_dir, f"X_val_{key}.npy"),   X_val_scaled)
        np.save(os.path.join(split_dir, f"y_val_{key}.npy"),   y_val)
        np.save(os.path.join(split_dir, f"X_test_{key}.npy"),  X_test_scaled)
        np.save(os.path.join(split_dir, f"y_test_{key}.npy"),  y_test)

        # Store in dictionary
        splits[key] = {
            "train": (X_train_t, y_train_t),
            "val":   (X_val_t,   y_val_t),
            "test":  (X_test_t,  y_test_t),
        }
        print(f"✅ Split done for {key} | Train: {X_train_scaled.shape}, Val: {X_val_scaled.shape}, Test: {X_test_scaled.shape}")

        # Alignment validation and save report alongside splits
        validation_results = validate_sequence_alignment(X, y, datetime_col.values, lookback=lb, horizon=hr)
        with open(os.path.join(split_dir, f"sequence_validation_lb{lb}_hr{hr}.json"), "w") as f:
            json.dump(validation_results, f, indent=2)

        # Per-lb/hr dataset configuration logging (using Fix 8)
        splits_dict = {
            "train": X_train_scaled,
            "val":   X_val_scaled,
            "test":  X_test_scaled,
        }
        _ = log_dataset_configuration(
            lookback=lb,
            horizon=hr,
            feature_set=f"allzones_{X_train_t.shape[-1]}features",
            splits=splits_dict,
            save_dir=split_dir
        )

💾 Split indices saved: assets\splits\split_indices_lb144_hr6.json
💾 Scaler saved: assets\scalers\scaler_X_allzones_lb144_hr6.joblib
💾 Scaler metadata saved: assets\scalers\scaler_metadata_lb144_hr6.json
✅ Split done for lb144_hr6 | Train: (36586, 144, 14), Val: (7840, 144, 14), Test: (7841, 144, 14)
🔍 Validating sequence alignment for lb144_hr6.
✅ Sequence alignment validation PASSED for lb144_hr6
   Total sequences: 52267
   Validation results saved: assets\prepared_sequences\sequence_validation_lb144_hr6.json
📊 Configuration logged for lb144_hr6 at assets\splits\config_log_lb144_hr6.json


### ✅ Step 3: Train/Val/Test Split, Scaling & Validation  

- Chronologically split data into 70/15/15 and saved split indices for reproducibility.  
- StandardScaler fit only on train features, applied to val/test, with artifacts (scaler + metadata) persisted.  
- Converted splits to tensors, logged dataset configs, and validated alignment to guard against leakage or timestamp gaps.  

In [18]:
# -----------------------------
# ✅ Step 4: DataLoader Preparation
# -----------------------------
from torch.utils.data import TensorDataset, DataLoader

batch_size = 64
dataloaders = {}

for key, sets in splits.items():
    X_train_t, y_train_t = sets["train"]
    X_val_t,   y_val_t   = sets["val"]
    X_test_t,  y_test_t  = sets["test"]

    # TensorDatasets
    train_ds = TensorDataset(X_train_t, y_train_t)
    val_ds   = TensorDataset(X_val_t,   y_val_t)
    test_ds  = TensorDataset(X_test_t,  y_test_t)

    # DataLoaders with drop_last=False for robustness
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  drop_last=False)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, drop_last=False)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, drop_last=False)

    dataloaders[key] = {
        "train": train_loader,
        "val":   val_loader,
        "test":  test_loader,
    }

    print(f"✅ DataLoaders ready for {key} | Batches -> "
          f"Train: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")

✅ DataLoaders ready for lb144_hr6 | Batches -> Train: 572, Val: 123, Test: 123


### ✅ Step 4: DataLoader Preparation  

- Wrapped train/validation/test splits into **TensorDataset** objects.  
- Built **PyTorch DataLoaders** with mini-batching (batch size = 64) and controlled shuffling.  
- Ensures efficient training, reproducible validation, and robust handling of sequence splits.  